# Visual reasoning по исходным пикселям

OpenCV извлекает кандидатов в объекты из цветовых масок и границ. Обучаемая модель оценивает их форму, цвет, материал, размер и положение. Исполнитель проверяет текстовые условия по этой таблице объектов.

Подключите данные соревнования и архив `visual-reasoning-pixels.zip` через Add Input. Подходит CPU или одна GPU. Internet можно выключить.

In [ ]:
RUN_FINAL = False
EPOCHS = 20
BATCH_SIZE = 64  # сцен
SEED = 42
BOUNDARIES = True
TRAIN_CROP_ANCHORS = True
INFERENCE = "joint"  # "joint", "blend" или "marginal"
DATA_DIR = None
SOURCE = None

## Данные и исходники

Все обучаемые параметры создаются с нуля. Фон оценивается только по train. Валидация обучает модель на train и считает метрики на отдельных val-сценах. Финальный запуск обучает модель на train с заданным числом эпох, затем открывает test и создаёт `final_submission.csv`.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
from zipfile import ZipFile

kaggle = Path("/kaggle/input").is_dir()
inputs = Path("/kaggle/input") if kaggle else Path.cwd()
output = Path("/kaggle/working") if kaggle else Path.cwd()

def one(explicit, matches, name):
    if explicit is not None:
        return Path(explicit).resolve(strict=True)
    options = sorted({p.resolve() for p in matches})
    if len(options) != 1:
        raise RuntimeError(f"Укажите {name} в первой ячейке. Найдены пути: {options}")
    return options[0]

candidates = [p.parent for p in inputs.rglob("train.csv")
              if (p.parent / "train").is_dir() and (p.parent / "val.csv").is_file()]
data = one(DATA_DIR, candidates, "DATA_DIR")
expanded = [p.parent.parent for p in inputs.rglob("pixel_reasoning.py")
            if (p.parent.parent / "src/pixel_regions.py").is_file()]
source = one(SOURCE, expanded or list(inputs.rglob("visual-reasoning-pixels.zip")), "SOURCE")
project = Path(tempfile.mkdtemp(prefix="visual_reasoning_pixels_"))
if source.is_dir():
    for folder in ("src", "scripts"):
        shutil.copytree(source / folder, project / folder, ignore=shutil.ignore_patterns("__pycache__"))
else:
    with ZipFile(source) as archive:
        if any(not (project / m).resolve().is_relative_to(project) for m in archive.namelist()):
            raise ValueError("В архиве найдены пути за пределами рабочей папки")
        archive.extractall(project)

subprocess.run([sys.executable, "-c", "import cv2, torch, numpy, scipy, sklearn"], check=True)
print(data)

## Обучение и оценка

Каждая пара описаний оценивается по одним и тем же объектам. При валидации отчёт содержит ROC-AUC, accuracy и ROC-AUC по категориям. Флаги обучения и seed сохраняются в отчёте.

In [ ]:
import json
import pandas as pd

receipt = output / "pixel_reasoning_report.json"
command = [
    sys.executable, "-u", "-m", "scripts.pixel_reasoning", "--data", str(data),
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--seed", str(SEED),
    "--report", str(receipt), "--output", str(output / "final_submission.csv"),
]
for active, flag in ((RUN_FINAL, "--final"), (BOUNDARIES, "--boundary"),
                     (TRAIN_CROP_ANCHORS, "--anchors")):
    if active:
        command.append(flag)
if INFERENCE != "marginal":
    command.extend(["--posterior", INFERENCE])
env = {**os.environ, "PYTHONHASHSEED": str(SEED), "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
       "OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}
subprocess.run(command, cwd=project, env=env, check=True)
report = json.loads(receipt.read_text(encoding="utf-8"))
if RUN_FINAL:
    print(report["submission"])
else:
    candidates = {key: report[key] for key in ("best_validation", "joint_validation", "blend_validation")
                  if key in report}
    display(pd.DataFrame.from_dict(candidates, orient="index").round(6))
print("Минуты:", report["minutes"])